<a href="https://colab.research.google.com/github/ChandrikaImmadi/GenAI-Tasks/blob/main/Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai faiss-cpu tiktoken

In [ ]:
from openai import OpenAI
import faiss
import numpy as np
import tiktoken

In [ ]:
# Import the OpenAI client (already done in a previous cell)
# from openai import OpenAI

# Securely retrieve your API key (e.g., from Colab secrets if applicable)
# For demonstration, we'll use the key already set, but in practice, use secrets.

# Initialize the client with your API key
# Note: A client named 'client' is already defined in this notebook.
# This example shows how you would initialize a new client if needed.
# my_new_openai_client = OpenAI(api_key="your_api_key_here")

# The existing 'client' variable in this notebook is already correctly initialized as:
# client = OpenAI(api_key="API_KEY")
# You can refer to cell `X7L3__Tv9s1D` for the working example in this notebook.

In [ ]:
openai.api_key = "API_KEY"

In [ ]:
client = OpenAI(api_key="API_KEY")
def get_embedding(text, model="text-embedding-3-small"):
    return client.embeddings.create(input=[text], model=model).data[0].embedding

In [ ]:
documents = [
    "The Eiffel Tower is located in Paris and was built in 1889.",
    "The Great Wall of China is a historic fortification stretching thousands of miles.",
    "The Colosseum in Rome was used for gladiatorial contests."
]

In [ ]:
dimension = len(get_embedding("test"))
index = faiss.IndexFlatL2(dimension)

In [ ]:
doc_embeddings = [get_embedding(doc) for doc in documents]
index.add(np.array(doc_embeddings).astype("float32"))

In [ ]:
def retrieve_chunks(query, k=2):
    query_emb = np.array(get_embedding(query)).astype("float32")
    D, I = index.search(np.array([query_emb]), k)
    return [documents[i] for i in I[0]]

In [ ]:
def answer_question(query):
    chunks = retrieve_chunks(query)
    context = "\n".join(chunks)

    prompt = f"""
    You are a helpful assistant. Use the following context to answer the question.

    Context:
    {context}

    Question: {query}
    Answer:
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content

In [ ]:
user_query = "Where is the Eiffel Tower located?"
print("=== User Query ===")
print(user_query)

print("\n=== Retrieved Chunks ===")
print(retrieve_chunks(user_query))

print("\n=== Final Answer ===")
print(answer_question(user_query))